In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from toxic_comments.config import HEAVY_TEXT_COLUMN, LABEL_COLUMNS, MODELS_DIR, RAW_DATA_DIR
from toxic_comments.cleaning import process_cleaning
from toxic_comments.repositories import CsvFileDatasetRepository, validate_training_data
from toxic_comments.folds import make_kfold_splits
from toxic_comments.evaluation import evaluate_predictions
from toxic_comments.models.roberta_label_dependency import build_roberta_label_dependency
from toxic_comments.predict import predict

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SRC_DIR exists:", SRC_DIR.exists())

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    print(f"✅ Dùng GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  KHÔNG CÓ GPU — đang chạy CPU, sẽ rất chậm (xem ước tính thời gian ở phần chat).")

DEVICE

In [ ]:
repository = CsvFileDatasetRepository(RAW_DATA_DIR / "train.csv")
data = process_cleaning(
    validate_training_data(repository.load()),
    is_train=True,
    verbose=False,
)
data = data[data["is_empty_heavy"] == 0].reset_index(drop=True)
print("Số dòng sau khi clean:", len(data))

In [ ]:
splits = make_kfold_splits(data, n_splits=5, random_state=42, text_column=HEAVY_TEXT_COLUMN)
train_index, test_index = splits[0]

X_train = data[HEAVY_TEXT_COLUMN].iloc[train_index]
y_train = data[LABEL_COLUMNS].iloc[train_index].to_numpy()
X_test = data[HEAVY_TEXT_COLUMN].iloc[test_index]
y_test = data[LABEL_COLUMNS].iloc[test_index].to_numpy()

print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
import time

model = build_roberta_label_dependency(device=DEVICE)  # ép dùng đúng device đã kiểm tra ở trên

start = time.perf_counter()
model.fit(X_train, y_train)
print(f"Train xong sau {time.perf_counter() - start:.1f}s")

In [ ]:
save_path = MODELS_DIR / "roberta_label_dependency"
model.save(save_path)
print("Đã lưu tại:", save_path)
print("Các file:", sorted(p.name for p in save_path.iterdir()))

In [ ]:
result_df = predict(
    list(X_test),
    model_dir=save_path,
    threshold=0.5,
    already_clean=True,  # X_test đã là comment_heavy rồi, khỏi clean lại
)

pred_cols = [f"{label}_pred" for label in LABEL_COLUMNS]
prob_cols = [f"{label}_prob" for label in LABEL_COLUMNS]
y_pred = result_df[pred_cols].to_numpy()
y_score = result_df[prob_cols].to_numpy()

metrics = evaluate_predictions(y_test, y_pred, y_score)
pd.Series(metrics, name="roberta_label_dependency (fold 1 test set, qua predict.py)")

In [ ]:
cv_results_path = PROJECT_ROOT / "results" / "cross_validation_results.csv"
if cv_results_path.exists():
    cv_results = pd.read_csv(cv_results_path)
    reference_row = cv_results[
        (cv_results["model_name"] == "roberta_label_dependency") & (cv_results["fold"] == 1)
    ]
    if not reference_row.empty:
        print("Fold 1 trong lần chạy CV trước:")
        print(reference_row[["macro_f1", "micro_f1"]].to_string(index=False))
    else:
        print("Chưa thấy fold 1 của roberta_label_dependency trong CSV — có thể CV chưa chạy xong tới đó.")
else:
    print(f"Chưa có {cv_results_path} — bỏ qua bước so sánh này.")